# Elasticsearch RAG

This notebook uses Elasticsearch as the retrieval store for the same RAG flow used in the persistent SQLite notebook.

In [1]:
import os
import time

import requests
from openai import OpenAI

from ingest import load_faq_data
from rag_helper import RAGBase

ELASTIC_URL = os.getenv("ELASTIC_URL", "http://localhost:9200").rstrip("/")
INDEX_NAME = os.getenv("ELASTIC_INDEX_NAME", "faq")
COURSE = "llm-zoomcamp"

session = requests.Session()

for _ in range(30):
    try:
        response = session.get(ELASTIC_URL, timeout=5)
        if response.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        f"Elasticsearch is not reachable at {ELASTIC_URL}. Start Elasticsearch or set ELASTIC_URL to a running cluster."
    )


def ensure_index():
    index_config = {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
        },
        "mappings": {
            "properties": {
                "question": {"type": "text"},
                "section": {"type": "text"},
                "answer": {"type": "text"},
                "course": {"type": "keyword"},
            }
        },
    }

    if session.head(f"{ELASTIC_URL}/{INDEX_NAME}", timeout=10).status_code == 200:
        return

    documents = load_faq_data()
    docs_llm = [doc for doc in documents if doc["course"] == COURSE]

    response = session.put(f"{ELASTIC_URL}/{INDEX_NAME}", json=index_config, timeout=30)
    response.raise_for_status()

    for i, doc in enumerate(docs_llm):
        response = session.post(
            f"{ELASTIC_URL}/{INDEX_NAME}/_doc/{i}",
            json=doc,
            timeout=30,
        )
        response.raise_for_status()

    response = session.post(f"{ELASTIC_URL}/{INDEX_NAME}/_refresh", timeout=30)
    response.raise_for_status()


ensure_index()


class ElasticsearchTextSearchIndex:
    def __init__(self, base_url, index_name, course=COURSE, session=None):
        self.base_url = base_url.rstrip("/")
        self.index_name = index_name
        self.course = course
        self.session = session or requests.Session()

    def _index_exists(self):
        response = self.session.head(f"{self.base_url}/{self.index_name}", timeout=10)
        return response.status_code == 200

    def count(self):
        if not self._index_exists():
            raise RuntimeError(f"Elasticsearch index '{self.index_name}' does not exist. Run the ingest notebook first.")

        response = self.session.get(
            f"{self.base_url}/{self.index_name}/_count",
            json={"query": {"term": {"course": self.course}}},
            timeout=30,
        )
        response.raise_for_status()
        return response.json()["count"]

    def search(self, query, num_results=5, boost_dict=None, filter_dict=None):
        if not self._index_exists():
            raise RuntimeError(f"Elasticsearch index '{self.index_name}' does not exist. Run the ingest notebook first.")

        boost_dict = boost_dict or {}
        filter_dict = filter_dict or {}

        fields = [
            f"question^{boost_dict.get('question', 3.0)}",
            f"section^{boost_dict.get('section', 0.5)}",
            f"answer^{boost_dict.get('answer', 1.0)}",
        ]

        filters = []
        course = filter_dict.get("course", self.course)
        if course is not None:
            filters.append({"term": {"course": course}})

        response = self.session.post(
            f"{self.base_url}/{self.index_name}/_search",
            json={
                "size": num_results,
                "query": {
                    "bool": {
                        "must": {
                            "multi_match": {
                                "query": query,
                                "fields": fields,
                            }
                        },
                        "filter": filters,
                    }
                },
            },
            timeout=30,
        )
        response.raise_for_status()
        return [hit["_source"] for hit in response.json()["hits"]["hits"]]

    def close(self):
        self.session.close()


es_index = ElasticsearchTextSearchIndex(ELASTIC_URL, INDEX_NAME, session=session)

In [2]:
es_index.count()

79

In [3]:
results = es_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 "OpenSource: I am using Groq, and it doesn't provide a tokenizer library based on my research. How can we estimate the number of OpenAI tokens asked in homework question 6?"]

In [4]:
openai_client = OpenAI()

assistant = RAGBase(
    index=es_index,
    llm_client=openai_client,
)

In [5]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, you need to submit your project while submissions are still open.


In [6]:
es_index.close()